# Sweep — Learning rate

Proxy fold 0 with gamma=1, wd=1e-4. 5 learning rates x 5 epochs. Winner: 1e-3.


In [1]:
from google.colab import drive
import os
import sys
from pathlib import Path

drive.mount("/content/drive")

REPRO_ROOT = Path("/content/drive/MyDrive/BirdCLEF_Project/repro")
if not (REPRO_ROOT / "birdclef").exists():
    raise FileNotFoundError(f"Expected repro at {REPRO_ROOT}")

sys.path.insert(0, str(REPRO_ROOT))
os.chdir(REPRO_ROOT)

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn matplotlib seaborn kaggle

from birdclef.colab import colab_prepare_training

repro_root = colab_prepare_training(stage_tta=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
train.csv already at /content/train.csv
sample_submission.csv already at /content/sample_submission.csv
Baseline embeddings already staged at /content/embeddings_v2
TTA embeddings already staged at /content/embeddings_v2_TTA
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
from birdclef.paths import MODELS_DIR

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df_TTA, NUM_CLASSES, _ = prepare_tta_data()
PROXY_FOLD = 0
EPOCHS = 5


In [4]:
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import BirdDataset, BirdMoE, FocalLoss, competition_macro_roc_auc, seed_everything
from birdclef.paths import SWEEPS_DIR
from birdclef.sweep_plots import plot_lr_sweep

SAVE_DIR = SWEEPS_DIR / "learning_rate"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

WINNING_GAMMA = 1.0
WINNING_WD = 1e-4
LR_VALUES = [1e-4, 3e-4, 5e-4, 1e-3, 3e-3]
train_df = df_TTA[df_TTA["fold"] != PROXY_FOLD].reset_index(drop=True)
valid_df = df_TTA[df_TTA["fold"] == PROXY_FOLD].reset_index(drop=True)
train_loader = DataLoader(BirdDataset(train_df, is_tta=True), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(BirdDataset(valid_df, is_tta=True), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

sweep_results = {}
sweep_history = {}

for current_lr in LR_VALUES:
    print(f"Testing lr={current_lr}")
    seed_everything(42)
    model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)
    criterion = FocalLoss(gamma=WINNING_GAMMA)
    optimizer = optim.Adam(model.parameters(), lr=current_lr, weight_decay=WINNING_WD)
    best_auc = 0.0
    val_aucs = []

    for epoch in range(EPOCHS):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            if torch.isnan(inputs).any():
                continue
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        val_aucs.append(current_auc)
        best_auc = max(best_auc, current_auc)
        print(f"  Epoch {epoch + 1}/{EPOCHS} | Val AUC: {current_auc:.4f}")

    sweep_results[current_lr] = best_auc
    sweep_history[current_lr] = {"val_aucs": val_aucs}
    print(f"Peak AUC for lr={current_lr}: {best_auc:.4f}")

best_lr = max(sweep_results, key=sweep_results.get)
print(f"Winner: lr={best_lr} (AUC={sweep_results[best_lr]:.4f})")
plot_lr_sweep(sweep_results, sweep_history, str(SAVE_DIR))


100%|██████████| 7110/7110 [00:09<00:00, 785.33it/s]


Testing lr=0.0001
  Epoch 1/5 | Val AUC: 0.9733
  Epoch 2/5 | Val AUC: 0.9744
  Epoch 3/5 | Val AUC: 0.9741
  Epoch 4/5 | Val AUC: 0.9741
  Epoch 5/5 | Val AUC: 0.9737
Peak AUC for lr=0.0001: 0.9744
Testing lr=0.0003
  Epoch 1/5 | Val AUC: 0.9750
  Epoch 2/5 | Val AUC: 0.9750
  Epoch 3/5 | Val AUC: 0.9747
  Epoch 4/5 | Val AUC: 0.9754
  Epoch 5/5 | Val AUC: 0.9741
Peak AUC for lr=0.0003: 0.9754
Testing lr=0.0005
  Epoch 1/5 | Val AUC: 0.9770
  Epoch 2/5 | Val AUC: 0.9765
  Epoch 3/5 | Val AUC: 0.9762
  Epoch 4/5 | Val AUC: 0.9761
  Epoch 5/5 | Val AUC: 0.9765
Peak AUC for lr=0.0005: 0.9770
Testing lr=0.001
  Epoch 1/5 | Val AUC: 0.9754
  Epoch 2/5 | Val AUC: 0.9752
  Epoch 3/5 | Val AUC: 0.9762
  Epoch 4/5 | Val AUC: 0.9775
  Epoch 5/5 | Val AUC: 0.9778
Peak AUC for lr=0.001: 0.9778
Testing lr=0.003
  Epoch 1/5 | Val AUC: 0.9750
  Epoch 2/5 | Val AUC: 0.9742
  Epoch 3/5 | Val AUC: 0.9752
  Epoch 4/5 | Val AUC: 0.9747
  Epoch 5/5 | Val AUC: 0.9763
Peak AUC for lr=0.003: 0.9763
Winner: l